# Weaver SDK 交互式演示

本 notebook 以 **Pig Latin** 翻译任务为例，演示 Weaver SDK 的核心原理以及每个操作背后发生了什么。

### 整体架构

```
┌────────────────────┐         HTTP/REST          ┌──────────────────────┐
│   你的代码 (SDK)    │  ◄──────────────────────►  │   Weaver 服务端       │
│                    │                             │                      │
│  ServiceClient     │   create_model ──────────►  │  Provisioner (调度)   │
│    └─ TrainingClient│   forward_backward ─────►  │    └─ Trainer (GPU)  │
│    └─ SamplingClient│   optim_step ───────────►  │    └─ Inference (GPU)│
│                    │   save_state / load_state ► │    └─ 存储系统        │
└────────────────────┘                             └──────────────────────┘
```

**核心理解**: SDK 是一个轻量的 HTTP 客户端。所有重计算（前向/反向传播、优化器更新、权重存储）都发生在**服务端 GPU** 上。SDK 在本地编排训练循环。

### 本 notebook 覆盖内容

| # | 主题 | 服务端发生了什么 |
|---|------|----------------|
| 1 | 数据准备 | 将文本 tokenize 为 `Datum` 对象 |
| 2 | LoRA 训练 | 服务端加载基座模型 + LoRA adapter 到 GPU |
| 3 | 全量微调 (Full FT) | 服务端加载完整模型权重（所有参数可训练） |
| 4 | 采样客户端 | 导出权重到推理引擎，生成文本 |
| 5 | save_state / load_state | Checkpoint 管理，支持训练恢复 |

---
## 安装 Weaver SDK

这会同时安装以下依赖：`torch`、`transformers`、`httpx` 等。

In [21]:
%pip install nex-weaver --upgrade

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


---
## 0. 环境配置

In [22]:
import os
from typing import Any, Dict, List, Tuple, Sequence

import torch

from weaver import ServiceClient, types

# 设置 API Key（或在启动 jupyter 前 export WEAVER_API_KEY）
API_KEY = os.getenv("WEAVER_API_KEY")
BASE_MODEL = "Qwen/Qwen3-8B"

assert API_KEY, "请设置 WEAVER_API_KEY 环境变量"

---
## 1. 数据准备

Weaver 以 **token 级别** 的数据为输入。每条训练样本是一个 `Datum`，包含：
- `model_input`: 喂给模型的输入 token 序列
- `loss_fn_inputs`: 传给服务端 loss 函数的 tensor 字典（如 target tokens、逐 token 的 weight）

对于 SFT，我们使用标准的 **next-token prediction** 构造方式：
- `input_tokens = tokens[:-1]`（去掉最后一个 token）
- `target_tokens = tokens[1:]`（右移一位）
- `weights`: prompt 部分为 0.0（不计算 loss），completion 部分为 1.0

In [23]:
EXAMPLES: List[Dict[str, str]] = [
    {"input": "banana split", "output": "anana-bay plit-say"},
    {"input": "quantum physics", "output": "uantum-qay ysics-phay"},
    {"input": "donut shop", "output": "onut-day op-shay"},
    {"input": "pickle jar", "output": "ickle-pay ar-jay"},
    {"input": "space exploration", "output": "ace-spay exploration-way"},
    {"input": "rubber duck", "output": "ubber-ray uck-day"},
    {"input": "coding wizard", "output": "oding-cay izard-way"},
]


def process_example(example: Dict[str, str], tokenizer) -> types.Datum:
    """将一对文本转换为 Weaver 的 token 级别 Datum。"""
    prompt = f"English: {example['input']}\nPig Latin:"
    prompt_tokens = tokenizer.encode(prompt, add_special_tokens=True)
    completion_tokens = tokenizer.encode(f" {example['output']}\n\n", add_special_tokens=False)

    tokens = prompt_tokens + completion_tokens
    # prompt 部分 weight=0（不计算 loss），completion 部分 weight=1
    weights = [0.0] * len(prompt_tokens) + [1.0] * len(completion_tokens)

    # 标准 next-token prediction 移位
    input_tokens = tokens[:-1]
    target_tokens = tokens[1:]
    weights = weights[1:]

    return types.Datum(
        model_input=types.ModelInput.from_ints(input_tokens),
        loss_fn_inputs={
            "target_tokens": torch.tensor(target_tokens, dtype=torch.int64),
            "weights": torch.tensor(weights, dtype=torch.float32),
        },
    )


def visualize_datum(datum: types.Datum, tokenizer) -> None:
    """展示 token 级别的 input / target / weight 对齐关系。"""
    print(f"{'Input':<20} {'Target':<20} {'Weight':<10}")
    print("-" * 50)
    for inp, tgt, wgt in zip(
        datum.model_input.to_ints(),
        datum.loss_fn_inputs["target_tokens"].tolist(),
        datum.loss_fn_inputs["weights"].tolist(),
    ):
        marker = "  ◄── 在此计算 loss" if wgt > 0 else ""
        print(f"{repr(tokenizer.decode([inp])):<20} {repr(tokenizer.decode([tgt])):<20} {wgt:<10}{marker}")

In [24]:
def extract_logprobs(output: Dict[str, Any]) -> torch.Tensor:
    """从 forward/backward 返回结果中提取 logprobs。"""
    value = output.get("logprobs") or output.get("Logprobs")
    if isinstance(value, dict):
        value = value.get("data")
    if value is None:
        raise ValueError("forward/backward 返回结果中缺少 logprobs")
    return torch.as_tensor(value, dtype=torch.float32)


def compute_loss(fwdbwd_result: dict, processed_examples: list) -> float:
    """根据 forward/backward 返回结果，在本地计算加权交叉熵 loss（用于监控）。"""
    outputs = fwdbwd_result.get("result", {}).get("loss_fn_outputs") or []
    logprobs = torch.cat([extract_logprobs(o) for o in outputs], dim=0)
    weights = torch.cat(
        [ex.loss_fn_inputs["weights"] for ex in processed_examples], dim=0
    )
    return float(-torch.dot(logprobs, weights) / weights.sum())

---
## 2. LoRA 训练

LoRA (Low-Rank Adaptation) 冻结基座模型权重，注入小规模可训练矩阵。

### 调用 `create_model()` 时服务端发生了什么（默认 = LoRA）：
1. SDK 发送 `POST /api/v1/sessions/{session_id}/models`，携带 `lora_config`
2. 服务端分配一个 **Trainer GPU 实例**
3. 加载基座模型权重，初始化 LoRA adapter（只有这些参数可训练）
4. 返回一个 `TrainingClient` 句柄

### LoRA 配置项：
- `rank`: 低秩矩阵的维度（默认=32）
- `train_attn`: 是否在 attention 层加 LoRA（默认=True）
- `train_mlp`: 是否在 MLP/MoE 层加 LoRA（默认=True）
- `train_unembed`: 是否在输出投影层加 LoRA（默认=True）
- `seed`: 用于 LoRA 权重初始化的随机种子

In [25]:
service_client = ServiceClient(api_key=API_KEY)
service_client.connect()
print("已连接到 Weaver 服务。")

已连接到 Weaver 服务。


In [27]:
# create_model() 默认 training_mode → LoRA (rank=32)
# 可自定义: lora_config=types.LoraConfig(rank=16, train_attn=True, train_mlp=False)
#
# 服务端行为:
#   POST /api/v1/sessions/{session_id}/models
#   Body: {base_model: "Qwen/Qwen3-8B", lora_config: {rank: 32, ...}}
#   → 服务端分配 Trainer GPU，加载模型 + LoRA adapter

lora_client = service_client.create_model(
    base_model=BASE_MODEL,
    lora_config=types.LoraConfig(rank=32),  # 这是默认值，这里显式写出来方便理解
)
print(f"Model ID: {lora_client.model_id}")
print(f"基座模型: {lora_client.base_model}")

Model ID: d1393404-3d93-4a88-af22-e034b2b49a21
基座模型: Qwen/Qwen3-8B


In [28]:
# get_tokenizer() 从 HuggingFace 下载并缓存 tokenizer
tokenizer = lora_client.get_tokenizer()

processed_examples = [process_example(ex, tokenizer) for ex in EXAMPLES]
print(f"已准备 {len(processed_examples)} 条训练样本\n")

# 可视化第一条样本的 token 级别结构
visualize_datum(processed_examples[0], tokenizer)

已准备 7 条训练样本

Input                Target               Weight    
--------------------------------------------------
'English'            ':'                  0.0       
':'                  ' banana'            0.0       
' banana'            ' split'             0.0       
' split'             '\n'                 0.0       
'\n'                 'P'                  0.0       
'P'                  'ig'                 0.0       
'ig'                 ' Latin'             0.0       
' Latin'             ':'                  0.0       
':'                  ' an'                1.0         ◄── 在此计算 loss
' an'                'ana'                1.0         ◄── 在此计算 loss
'ana'                '-b'                 1.0         ◄── 在此计算 loss
'-b'                 'ay'                 1.0         ◄── 在此计算 loss
'ay'                 ' pl'                1.0         ◄── 在此计算 loss
' pl'                'it'                 1.0         ◄── 在此计算 loss
'it'                 '-s'                 1.0      

### forward_backward + optim_step = 一个 mini-batch

在传统训练框架中，一个 mini-batch 步骤包含：forward → loss → backward → optimizer.step()。

Weaver 的语义完全对应：

```
┌─────────────────────────────────────────────────────────────────┐
│  forward_backward(data, "cross_entropy")                        │
│                                                                 │
│    SDK 发送 data (N 条 Datum) ──►  Server (Trainer GPU)         │
│                                                                 │
│    Server 内部:                                                 │
│      1. 将 N 条 Datum 自动拆成 micro-batches                    │
│         (受 GPU 显存限制，每个 micro-batch 单独做 fwd+bwd)        │
│      2. 梯度在所有 micro-batches 上累加 (gradient accumulation)  │
│      3. 返回每条 Datum 的 logprobs 给 SDK (用于本地 loss 监控)   │
│                                                                 │
│    ⚠️ 注意: forward_backward 不会拆 mini-batch!                 │
│    你传入的 data 就是整个 mini-batch，Server 只做 micro-batch 拆分│
│    如果需要多个 mini-batch 组成一个大 batch，需要在上层控制       │
│    (例如 NexRL 框架负责 mini-batch 的编排)                       │
├─────────────────────────────────────────────────────────────────┤
│  optim_step(adam_params)                                        │
│                                                                 │
│    Server 内部:                                                 │
│      1. 用累加的梯度做一次 Adam 更新                             │
│      2. 清零梯度，准备下一个 mini-batch                          │
│                                                                 │
│    ⚠️ 一次 forward_backward + 一次 optim_step = 一次权重更新    │
└─────────────────────────────────────────────────────────────────┘
```

**与传统训练的对应关系:**

| 传统训练概念 | Weaver 对应 | 谁负责 |
|------------|------------|--------|
| mini-batch (N 条数据) | `forward_backward(data)` 中的 `data` | 用户 / NexRL |
| micro-batch (显存拆分) | Server 自动拆分 | Weaver Server |
| gradient accumulation across micro-batches | Server 自动完成 | Weaver Server |
| 多次 mini-batch 组成大 batch | 多次 `forward_backward` 后再 `optim_step` | 用户 / NexRL |
| optimizer.step() | `optim_step(adam_params)` | Weaver Server |

**进阶: 梯度累积多个 mini-batch**

如果你想在多个 mini-batch 上累积梯度再更新（等价于更大的 effective batch size），可以调用多次 `forward_backward` 再调一次 `optim_step`:

```python
# 等效 effective_batch = mini_batch_1 + mini_batch_2 + mini_batch_3
for mini_batch in [mini_batch_1, mini_batch_2, mini_batch_3]:
    training_client.forward_backward(mini_batch, "cross_entropy", wait=True)

# 梯度在 Server 上已经跨 3 次 forward_backward 累加
training_client.optim_step(adam, wait=True)  # 一次权重更新
```

In [29]:
# 每一轮: forward_backward + optim_step = 传统训练的一个 mini-batch step
#
# processed_examples (7 条) 就是本次 mini-batch
# Server 内部会自动拆 micro-batch 处理，对用户透明

adam = types.AdamParams(learning_rate=1e-4)

print("=== LoRA 训练（每 step = 1 个 mini-batch）===")
for step in range(3):
    # forward + loss + backward: 服务端自动拆 micro-batch，梯度累加
    fwdbwd_result = lora_client.forward_backward(
        processed_examples,  # ← 这就是一个完整的 mini-batch
        "cross_entropy",
        wait=True,
    )
    # optimizer.step(): 用累加的梯度做一次 Adam 更新，然后清零梯度
    lora_client.optim_step(adam, wait=True)

    loss = compute_loss(fwdbwd_result, processed_examples)
    print(f"Step {step}: loss/token = {loss:.4f}")

print("\nLoRA 训练完成。")

=== LoRA 训练（每 step = 1 个 mini-batch）===
Step 0: loss/token = 4.2465
Step 1: loss/token = 3.4124
Step 2: loss/token = 2.5615

LoRA 训练完成。


您要不 PUT 一下这个模型捏：upsert

curl -X PUT https://weaver-console.nex-agi.cn/api/v1/supported-models \
  -H "Content-Type: application/json" \
  -H "X-WEAVER-API-KEY: sk-e1ebbd918ee5b0fd1ac3cc0e7c9654883e6118e676dd9184347ff11c73ae44eb" \
  -d '{
            "name": "Tian/Qwen3-8B-dev",
            "config": {
                "resource": {
                    "train": {
                        "world_size": 1,
                        "gpus_per_pod": 8,
                        "sp_size": 8,
                        "use_ulysses": 1,
                        "use_ring": 0,
                        "memory_per_gpu": 200
                    },
                    "inference": {
                        "backend": "bp-sglang",
                        "tp_size": 2,
                        "replicas": 4,
                        "gpus_per_replica": 2
                    },
                    "tokenizer": {
                        "path": "/gpfs/models/huggingface.co/Qwen/Qwen3-8B"
                    }
                },
                "job_creator": "tangtian",
                "base_model_path": "/gpfs/models/huggingface.co/Qwen/Qwen3-8B",
                "rollout_tp_size": 2
            }
        }'

---
## 3. 全量微调 (Full Fine-Tuning)

全量微调让**所有**模型参数都可训练（不使用 LoRA adapter）。

### 与 LoRA 的区别：
- `training_mode="full_ft"` → 服务端不创建 LoRA adapter
- 所有 transformer 权重直接被优化器更新
- 通常需要更**小的学习率**（如 1e-5）以避免灾难性遗忘
- 服务端需要更多 GPU 显存

In [31]:
# 创建全量微调模型
#
# 服务端行为:
#   POST /api/v1/sessions/{session_id}/models
#   Body: {base_model: "Qwen/Qwen3-8B", training_mode: "full_ft"}
#   → 不发送 lora_config；服务端让所有权重可训练

fullft_client = service_client.create_model(
    base_model=BASE_MODEL,
    training_mode="full_ft",
)
print(f"全量微调 Model ID: {fullft_client.model_id}")

全量微调 Model ID: 7b040314-3cee-4f7e-9ea5-8e7639a8ccd7


In [32]:
tokenizer_ft = fullft_client.get_tokenizer()
processed_examples_ft = [process_example(ex, tokenizer_ft) for ex in EXAMPLES]

# 全量微调用更小的学习率
adam_ft = types.AdamParams(learning_rate=1e-5)

print("=== 全量微调 ===")
for step in range(3):
    fwdbwd_result = fullft_client.forward_backward(
        processed_examples_ft, "cross_entropy", wait=True
    )
    fullft_client.optim_step(adam_ft, wait=True)

    loss = compute_loss(fwdbwd_result, processed_examples_ft)
    print(f"Step {step}: loss/token = {loss:.4f}")

print("\n全量微调完成。")

=== 全量微调 ===
Step 0: loss/token = 4.2465
Step 1: loss/token = 2.1362
Step 2: loss/token = 0.3820

全量微调完成。


---
## 4. 采样客户端 (Sampling Client)

训练完成后，需要将训练好的权重**导出**到推理引擎来生成文本。

### 服务端发生了什么：
1. `save_weights_and_get_sampling_client()` → POST 导出请求到服务端
2. 服务端将训练好的权重（LoRA 合并或全量）转换为推理优化格式
3. 创建一个推理 GPU 会话
4. 返回一个 `SamplingClient`，可通过 `sample()` 生成文本

### 也可以分两步操作：
```python
model_path = training_client.save_weights_for_sampler(name="my-model")
sampling_client = service_client.create_sampling_client(base_model=..., model_path=model_path)
```

In [33]:
# 一步完成：导出 LoRA 权重 + 获取采样客户端
#
# 服务端行为:
#   POST /api/v1/models/{model_id}/export-sampler
#   → 服务端将 LoRA 权重合并到基座模型（如果是 LoRA）
#   → 创建推理会话，加载合并后的权重
#   → 返回 sampling_session_id + model_path

sampling_client = lora_client.save_weights_and_get_sampling_client(
    name="pig-latin-lora-model"
)
print(f"采样客户端就绪。模型路径: {sampling_client.model_path}")

采样客户端就绪。模型路径: weaver://d1393404-3d93-4a88-af22-e034b2b49a21/pig-latin-lora-model


In [34]:
# 用训练好的模型生成 Pig Latin 翻译
#
# sample() 将 prompt 发送到推理引擎，返回生成的 token 序列
# SamplingParams 控制解码策略: temperature, top_p, stop tokens 等

test_inputs = ["coffee break", "hello world", "machine learning"]

for text in test_inputs:
    prompt_str = f"English: {text}\nPig Latin:"
    prompt_tokens = tokenizer.encode(prompt_str, add_special_tokens=True)
    prompt = types.ModelInput.from_ints(prompt_tokens)

    params = types.SamplingParams(
        max_tokens=20,
        temperature=0.0,   # 贪心解码
        stop=["\n"],       # 遇到换行停止
    )
    result = sampling_client.sample(
        prompt=prompt,
        sampling_params=params,
        num_samples=1,
    )
    decoded = tokenizer.decode(result["sequences"][0].get("tokens", []))
    print(f"{text:>20}  →  {decoded.strip()}")

        coffee break  →  oofay ebray akebray
         hello world  →  olay helloworlday
    machine learning  →  eanm aelre egnil


---
## 5. Checkpoint 管理: save_state / load_state

Weaver 支持保存和加载模型 checkpoint，用于训练恢复。

### Checkpoint 类型：
- `"weight"`（默认）: 仅保存模型权重
- `"weight_and_optimizer"`: 保存权重 + Adam 动量/方差状态

### API 方法：
| 方法 | 功能 |
|------|------|
| `save_state()` | 服务端将 checkpoint 写入存储，返回 `Checkpoint` 对象 |
| `load_state(path)` | 仅恢复权重（优化器状态重置） |
| `load_state_with_optimizer(path)` | 恢复权重 + 优化器状态（真正的断点续训） |
| `list_checkpoints()` | 列出该模型的所有 checkpoint |

### 服务端行为：
```
save_state(name="step-6"):
  POST /api/v1/models/{model_id}/checkpoints
  → Trainer 将权重写入: weaver://{model_id}/checkpoints/step-6
  → 返回 Checkpoint(id=..., path="weaver://...", name="step-6")

load_state(checkpoint):
  POST /api/v1/models/{model_id}/load
  Body: {path: "weaver://...", include_optimizer: false}
  → Trainer 从存储加载权重，替换当前权重
```

In [35]:
# 保存仅权重的 checkpoint
ckpt_weight = lora_client.save_state(name="lora-step-9", checkpoint_type="weight")
print(f"已保存 checkpoint:")
print(f"  id:   {ckpt_weight.id}")
print(f"  path: {ckpt_weight.path}")
print(f"  name: {ckpt_weight.name}")
print(f"  type: {ckpt_weight.checkpoint_type}")

已保存 checkpoint:
  id:   d3cbb2a8-a784-4b3b-9b43-7e8a9c316107
  path: weaver://d1393404-3d93-4a88-af22-e034b2b49a21/checkpoints/lora-step-9
  name: lora-step-9
  type: weight


In [36]:
# 保存包含优化器状态的 checkpoint（用于断点续训）
ckpt_full = lora_client.save_state(
    name="lora-step-9-with-optim",
    checkpoint_type="weight_and_optimizer",
)
print(f"已保存完整 checkpoint: {ckpt_full.path}")

已保存完整 checkpoint: weaver://d1393404-3d93-4a88-af22-e034b2b49a21/checkpoints/lora-step-9-with-optim


In [37]:
# 列出该模型的所有 checkpoint
checkpoints = lora_client.list_checkpoints()
print(f"共 {len(checkpoints)} 个 checkpoint:")
for ckpt in checkpoints:
    print(f"  [{ckpt.checkpoint_type:>22}] {ckpt.name or '(未命名)':>30}  →  {ckpt.path}")

共 4 个 checkpoint:
  [  weight_and_optimizer]         lora-step-9-with-optim  →  weaver://d1393404-3d93-4a88-af22-e034b2b49a21/checkpoints/lora-step-9-with-optim
  [                weight]                    lora-step-9  →  weaver://d1393404-3d93-4a88-af22-e034b2b49a21/checkpoints/lora-step-9
  [              sampling]                          (未命名)  →  weaver://d1393404-3d93-4a88-af22-e034b2b49a21/pig-latin-lora-model
  [              sampling]                          (未命名)  →  weaver://d1393404-3d93-4a88-af22-e034b2b49a21/pig-latin-lora-model


In [38]:
# load_state: restore weights only (optimizer state is reset)
# Use case: roll back to an earlier checkpoint, then continue training with fresh optimizer
lora_client.load_state(ckpt_weight)
print(f"Loaded weights from: {ckpt_weight.path}")
print("Optimizer state was reset.")

Loaded weights from: weaver://d1393404-3d93-4a88-af22-e034b2b49a21/checkpoints/lora-step-9
Optimizer state was reset.


In [39]:
# load_state_with_optimizer: true resume (weights + Adam momentum/variance)
# Use case: resume training exactly where you left off
lora_client.load_state_with_optimizer(ckpt_full)
print(f"Loaded weights + optimizer from: {ckpt_full.path}")
print("Training can resume with preserved optimizer state.")

Loaded weights + optimizer from: weaver://d1393404-3d93-4a88-af22-e034b2b49a21/checkpoints/lora-step-9-with-optim
Training can resume with preserved optimizer state.


In [40]:
# Continue training after loading checkpoint
print("=== Resumed Training ===")
adam_resumed = types.AdamParams(learning_rate=1e-4)
for step in range(3):
    fwdbwd_result = lora_client.forward_backward(
        processed_examples, "cross_entropy", wait=True
    )
    lora_client.optim_step(adam_resumed, wait=True)

    loss = compute_loss(fwdbwd_result, processed_examples)
    print(f"Resumed step {step}: loss/token = {loss:.4f}")

print("\nResumed training complete.")

=== Resumed Training ===
Resumed step 0: loss/token = 1.9873
Resumed step 1: loss/token = 1.4934
Resumed step 2: loss/token = 1.0437

Resumed training complete.


---
## 6. Cleanup

`terminate()` releases the GPU instances (trainer + inference) provisioned for each model. The `ServiceClient` context manager also sends a session close on exit.

In [41]:
lora_client.terminate()
fullft_client.terminate()
service_client.close()
print("All resources released.")

All resources released.


---
## Quick Reference

| SDK Call | Server-Side Effect |
|----------|--------------------|
| `ServiceClient(api_key=...)` | Establishes authenticated session |
| `create_model(base_model=..., training_mode=None)` | Provisions GPU, loads model + LoRA adapters |
| `create_model(base_model=..., training_mode="full_ft")` | Provisions GPU, loads model (all params trainable) |
| `forward_backward(data, "cross_entropy")` | Forward pass + loss + backward pass on GPU |
| `optim_step(AdamParams(...))` | Adam update on accumulated gradients |
| `save_weights_and_get_sampling_client()` | Export weights → create inference session |
| `sampling_client.sample(prompt, params)` | Generate text using trained model |
| `save_state(name=..., checkpoint_type=...)` | Write checkpoint to server storage |
| `load_state(checkpoint)` | Restore weights (optimizer reset) |
| `load_state_with_optimizer(checkpoint)` | Restore weights + optimizer (true resume) |
| `terminate()` | Release GPU resources |